# sprint 3. modelo 1: XGBoost — predicao de desfecho adverso binario

projeto: infant dysbiosis predictor  
sprint: 28/4  

etapas:
1. treinar XGBoost para predicao de desfecho adverso binario aos 2 anos (n=210, holdout 20%)
2. aplicar validacao cruzada estratificada para lidar com desbalanceamento de classes
3. avaliar com Recall, ROC-AUC e F1-Score
4. comparar desempenho com e sem a covariate `antibiotics_first_2_years`

## 0. imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_validate
)
from sklearn.metrics import (
    recall_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, RocCurveDisplay
)
from sklearn.dummy import DummyClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 30)
SEED = 42

## 1. carregar dataset

> **pre-requisito:** executar `sprint2_preprocessing.ipynb` para gerar `data/processed/model1_encoded.csv`.

In [ ]:
df = pd.read_csv('../data/processed/model1_encoded.csv')

TARGET = 'adverse_outcomes'
DROP   = ['sample_id', TARGET]
ALL_FEATURES = [c for c in df.columns if c not in DROP]

print(f'shape  : {df.shape}')
print(f'target : {TARGET}')
print(f'features ({len(ALL_FEATURES)}): {ALL_FEATURES}')
print()

pos = df[TARGET].sum()
neg = len(df) - pos
print(f'positivos: {pos} ({pos/len(df)*100:.1f}%)')
print(f'negativos: {neg} ({neg/len(df)*100:.1f}%)')
print(f'scale_pos_weight sugerido: {neg/pos:.2f}  (neg/pos)')

## 2. holdout estratificado 80/20

separamos 20% dos dados (n≈42) como holdout **antes de qualquer treino**.  
esse conjunto so sera tocado na avaliacao final — nunca durante a busca de hiperparametros ou CV.  
estratificado para manter a proporcao de positivos (~30%) em treino e teste.

In [ ]:
X_all = df[ALL_FEATURES].values
y_all = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.20,
    stratify=y_all,
    random_state=SEED
)

spw = (y_train == 0).sum() / (y_train == 1).sum()  # scale_pos_weight no treino

print(f'treino  : {X_train.shape[0]} amostras  '
      f'({y_train.sum()} pos, {(y_train==0).sum()} neg)')
print(f'holdout : {X_test.shape[0]} amostras  '
      f'({y_test.sum()} pos, {(y_test==0).sum()} neg)')
print(f'scale_pos_weight (treino): {spw:.2f}')

## 3. baseline

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy_cv = cross_validate(
    dummy, X_train, y_train, cv=cv5,
    scoring={'recall': 'recall', 'roc_auc': 'roc_auc', 'f1': 'f1'}
)

print('baseline (majoritario) — CV 5-fold no treino:')
for m in ['recall', 'roc_auc', 'f1']:
    vals = dummy_cv[f'test_{m}']
    print(f'  {m:<10}: {vals.mean():.4f}  ± {vals.std():.4f}')

## 4. validacao cruzada estratificada e desbalanceamento de classes\n\ncom apenas ~30% de positivos, uma CV nao-estratificada pode gerar folds onde a proporcao de positivos varia muito — o modelo aprende num fold com 20% de positivos e e avaliado num fold com 40%, tornando as metricas instáveis e nao comparaveis entre folds.\n\nessa secao demonstra:\n1. o impacto do desbalanceamento nas metricas sem qualquer correcao\n2. a diferenca entre CV estratificada e nao-estratificada\n3. o efeito de `scale_pos_weight` nas predicoes\n4. analise da distribuicao de classes em cada fold

### 4.1 visualizar o desbalanceamento

In [ ]:
from sklearn.model_selection import KFold

pos_rate_train = y_train.mean()
neg_rate_train = 1 - pos_rate_train

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# proporcao geral no treino
axes[0].bar(['negativo', 'positivo'],
            [(y_train == 0).sum(), y_train.sum()],
            color=['#1D9E75', '#E24B4A'], width=0.4)
for bar, n in zip(axes[0].patches,
                  [(y_train==0).sum(), y_train.sum()]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1, str(n),
                 ha='center', fontsize=11)
axes[0].set_title(f'distribuicao no treino (n={len(y_train)})')
axes[0].set_ylabel('n')

# % de positivos por fold — estratificada vs nao-estratificada
cv_strat    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_nostrat  = KFold(n_splits=5, shuffle=True, random_state=SEED)

pct_strat   = [y_train[va].mean() * 100 for _, va in cv_strat.split(X_train, y_train)]
pct_nostrat = [y_train[va].mean() * 100 for _, va in cv_nostrat.split(X_train, y_train)]

x = np.arange(5)
w = 0.35
axes[1].bar(x - w/2, pct_nostrat, w, label='KFold (nao-estratificada)',
            color='#AAAAAA', alpha=0.9)
axes[1].bar(x + w/2, pct_strat,   w, label='StratifiedKFold',
            color='#378ADD', alpha=0.9)
axes[1].axhline(pos_rate_train * 100, color='#E24B4A',
                linestyle='--', linewidth=1.5, label=f'real ({pos_rate_train*100:.1f}%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'fold {i+1}' for i in range(5)])
axes[1].set_ylabel('% positivos no fold de validacao')
axes[1].set_title('% positivos por fold: estratificada vs nao-estratificada')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 60)

# variancia entre folds
variances = {
    'KFold\n(nao-estrat)': np.std(pct_nostrat),
    'StratifiedKFold': np.std(pct_strat),
}
bars = axes[2].bar(variances.keys(), variances.values(),
                   color=['#AAAAAA', '#378ADD'], alpha=0.9, width=0.4)
axes[2].bar_label(bars, labels=[f'{v:.2f}pp' for v in variances.values()],
                  fontsize=11, padding=3)
axes[2].set_ylabel('desvio padrao da % de positivos por fold')
axes[2].set_title('variabilidade entre folds')

plt.suptitle('impacto da estratificacao na distribuicao de classes por fold',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('% de positivos por fold:')
print(f'  KFold         : {[f"{v:.1f}%" for v in pct_nostrat]}  std={np.std(pct_nostrat):.2f}pp')
print(f'  StratifiedKFold: {[f"{v:.1f}%" for v in pct_strat]}  std={np.std(pct_strat):.2f}pp')

### 4.2 efeito do scale_pos_weight nas predicoes\n\nsem `scale_pos_weight`, o XGBoost tende a prever quase tudo como negativo (classe majoritaria) — recall proximo de zero.  \ncom `scale_pos_weight = neg/pos`, a funcao de perda penaliza mais os erros nos positivos, aumentando o recall ao custo de mais falsos positivos.

In [ ]:
xgb_params = dict(n_estimators=200, max_depth=3, learning_rate=0.1,
                  subsample=0.8, colsample_bytree=0.8,
                  eval_metric='logloss', random_state=SEED, n_jobs=-1)

results_spw = {}
for spw_val in [1, spw]:
    label = f'spw={spw_val:.1f}' if spw_val != 1 else 'spw=1 (sem correcao)'
    scores = {'recall': [], 'precision': [], 'f1': [], 'roc_auc': []}

    for tr_idx, va_idx in cv5.split(X_train, y_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]

        m = XGBClassifier(**xgb_params, scale_pos_weight=spw_val)
        m.fit(X_tr, y_tr)
        y_p = m.predict(X_va)
        y_b = m.predict_proba(X_va)[:, 1]

        scores['recall'].append(recall_score(y_va, y_p, zero_division=0))
        scores['precision'].append(__import__('sklearn.metrics', fromlist=['precision_score'])
                                   .precision_score(y_va, y_p, zero_division=0))
        scores['f1'].append(f1_score(y_va, y_p, zero_division=0))
        scores['roc_auc'].append(roc_auc_score(y_va, y_b))

    results_spw[label] = {k: np.array(v) for k, v in scores.items()}

# plot
metrics = ['recall', 'precision', 'f1', 'roc_auc']
labels  = list(results_spw.keys())
x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
for i, (label, scores) in enumerate(results_spw.items()):
    means = [scores[m].mean() for m in metrics]
    stds  = [scores[m].std()  for m in metrics]
    offset = (i - 0.5) * w
    bars = ax.bar(x + offset, means, w, label=label,
                  yerr=stds, capsize=4, alpha=0.85,
                  color='#AAAAAA' if i == 0 else '#378ADD')
    ax.bar_label(bars, labels=[f'{v:.3f}' for v in means],
                 fontsize=8, padding=4)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.2)
ax.set_ylabel('score (media 5-fold)')
ax.set_title('efeito do scale_pos_weight — CV estratificada 5-fold')
ax.legend()
plt.tight_layout()
plt.show()

print('medias por metrica:')
for label, scores in results_spw.items():
    print(f'\n  {label}')
    for m in metrics:
        print(f'    {m:<12}: {scores[m].mean():.4f} ± {scores[m].std():.4f}')

### 4.3 analise do threshold de decisao\n\npor padrao o XGBoost usa threshold=0.5 para converter probabilidades em classes.  \ncom desbalanceamento, um threshold menor pode aumentar o recall sem destruir o F1.  \nessa analise encontra o threshold otimo no conjunto de treino (via CV) sem tocar o holdout.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)

thresh_recall = np.zeros(len(thresholds))
thresh_f1     = np.zeros(len(thresholds))
thresh_prec   = np.zeros(len(thresholds))

for tr_idx, va_idx in cv5.split(X_train, y_train):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    m = XGBClassifier(**xgb_params, scale_pos_weight=spw)
    m.fit(X_tr, y_tr)
    probs = m.predict_proba(X_va)[:, 1]

    for i, t in enumerate(thresholds):
        pred = (probs >= t).astype(int)
        from sklearn.metrics import precision_score
        thresh_recall[i] += recall_score(y_va, pred, zero_division=0)
        thresh_f1[i]     += f1_score(y_va, pred, zero_division=0)
        thresh_prec[i]   += precision_score(y_va, pred, zero_division=0)

thresh_recall /= 5
thresh_f1     /= 5
thresh_prec   /= 5

best_thresh_recall = thresholds[np.argmax(thresh_recall)]
best_thresh_f1     = thresholds[np.argmax(thresh_f1)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, thresh_recall, color='#E24B4A', marker='o', ms=4, label='recall')
ax.plot(thresholds, thresh_f1,     color='#378ADD', marker='s', ms=4, label='F1')
ax.plot(thresholds, thresh_prec,   color='#1D9E75', marker='^', ms=4, label='precision')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1,
           label='threshold padrao (0.5)')
ax.axvline(best_thresh_f1, color='#378ADD', linestyle=':',
           linewidth=1.5, label=f'melhor F1 ({best_thresh_f1:.2f})')
ax.set_xlabel('threshold de decisao')
ax.set_ylabel('score medio (5-fold CV)')
ax.set_title('recall, F1 e precision por threshold — CV estratificada')
ax.legend(fontsize=9)
ax.set_xlim(0.1, 0.9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'threshold padrao (0.5) → recall={thresh_recall[thresholds==0.5][0]:.4f}  '
      f'F1={thresh_f1[thresholds==0.5][0]:.4f}')
print(f'melhor threshold p/ F1  ({best_thresh_f1:.2f}) → '
      f'recall={thresh_recall[np.argmax(thresh_f1)]:.4f}  '
      f'F1={thresh_f1.max():.4f}')
print()
print('nota: o threshold otimo sera aplicado na avaliacao final do holdout')

BEST_THRESHOLD = best_thresh_f1

## 4. XGBoost com GridSearchCV aninhado (CV estratificada 5-fold)

protocolo identico ao modelo 2:
- CV interna 3-fold para busca de hiperparametros
- CV externa 5-fold para estimar performance
- tudo feito **apenas no conjunto de treino** (80%)
- `scale_pos_weight` fixo em cada fold para compensar desbalanceamento

In [ ]:
PARAM_GRID = {
    'n_estimators'  : [200, 400],
    'max_depth'     : [3, 5, 7],
    'learning_rate' : [0.05, 0.1],
    'subsample'     : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

cv_scores = {'recall': [], 'roc_auc': [], 'f1': [], 'best_params': []}

for fold, (tr_idx, va_idx) in enumerate(cv5.split(X_train, y_train), 1):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    spw_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    xgb = XGBClassifier(
        scale_pos_weight=spw_fold,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1
    )
    gs = GridSearchCV(xgb, PARAM_GRID, cv=inner_cv,
                      scoring='recall', n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)

    y_pred = gs.predict(X_va)
    y_prob = gs.predict_proba(X_va)[:, 1]

    rec = recall_score(y_va, y_pred)
    auc = roc_auc_score(y_va, y_prob)
    f1  = f1_score(y_va, y_pred)

    cv_scores['recall'].append(rec)
    cv_scores['roc_auc'].append(auc)
    cv_scores['f1'].append(f1)
    cv_scores['best_params'].append(gs.best_params_)

    print(f'  fold {fold}  recall={rec:.4f}  roc_auc={auc:.4f}  f1={f1:.4f}  '
          f'params={gs.best_params_}')

print()
for m in ['recall', 'roc_auc', 'f1']:
    vals = np.array(cv_scores[m])
    print(f'{m:<10}: {vals.mean():.4f}  ± {vals.std():.4f}')

## 5. modelo final: treinar no conjunto de treino completo (80%)

hiperparametros mais frequentes entre os 5 folds.

In [ ]:
from collections import Counter

best_params_final = {}
for k in PARAM_GRID:
    vals = [p[k] for p in cv_scores['best_params']]
    best_params_final[k] = Counter(vals).most_common(1)[0][0]

print('hiperparametros mais frequentes:', best_params_final)

xgb_final = XGBClassifier(
    **best_params_final,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1
)
xgb_final.fit(X_train, y_train)
print(f'\nmodelo final treinado (n_treino={len(X_train)})')

## 6. avaliacao no holdout (20%)

primeira e unica vez que o holdout e tocado.

In [ ]:
y_pred_test = xgb_final.predict(X_test)
y_prob_test = xgb_final.predict_proba(X_test)[:, 1]

rec_test = recall_score(y_test, y_pred_test)
auc_test = roc_auc_score(y_test, y_prob_test)
f1_test  = f1_score(y_test, y_pred_test)

print('=== avaliacao no holdout (20%, nunca visto no treino) ===\n')
print(f'Recall    : {rec_test:.4f}')
print(f'ROC-AUC   : {auc_test:.4f}')
print(f'F1-Score  : {f1_test:.4f}')
print()
print(classification_report(y_test, y_pred_test,
                             target_names=['negativo', 'positivo'], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# matriz de confusao
cm      = confusion_matrix(y_test, y_pred_test)
cm_norm = confusion_matrix(y_test, y_pred_test, normalize='true')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['negativo', 'positivo'],
            yticklabels=['negativo', 'positivo'],
            linewidths=0.5, cbar=False, annot_kws={'size': 13})
for i in range(2):
    for j in range(2):
        axes[0].text(j + 0.5, i + 0.72,
                     f'({cm_norm[i,j]*100:.0f}%)',
                     ha='center', va='center', fontsize=9, color='gray')
axes[0].set_title('matriz de confusao — holdout')
axes[0].set_xlabel('predito')
axes[0].set_ylabel('real')

# curva ROC
RocCurveDisplay.from_predictions(y_test, y_prob_test, ax=axes[1],
                                  name=f'XGBoost (AUC={auc_test:.3f})',
                                  color='#378ADD')
axes[1].plot([0,1],[0,1],'--',color='gray',label='chance')
axes[1].set_title('curva ROC — holdout')
axes[1].legend()

plt.suptitle('XGBoost — avaliacao no holdout (n=42)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 6.2 threshold otimizado vs padrao (0.5)\n\naplicamos o `BEST_THRESHOLD` encontrado na CV (secao 4.3) e comparamos com o threshold padrao.

In [ ]:
from sklearn.metrics import precision_score

y_pred_05   = (y_prob_test >= 0.50).astype(int)
y_pred_best = (y_prob_test >= BEST_THRESHOLD).astype(int)

def metrics_dict(y_true, y_pred, y_prob):
    return {
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_true, y_prob),
    }

m_05   = metrics_dict(y_test, y_pred_05,   y_prob_test)
m_best = metrics_dict(y_test, y_pred_best, y_prob_test)

# tabela comparativa
comp = pd.DataFrame({'threshold=0.50': m_05, f'threshold={BEST_THRESHOLD:.2f} (otimizado)': m_best}).T
print(comp.round(4).to_string())

# plot
metric_names = list(m_05.keys())
x = np.arange(len(metric_names))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - w/2, [m_05[m]   for m in metric_names], w,
            label='threshold=0.50', color='#AAAAAA', alpha=0.9)
b2 = ax.bar(x + w/2, [m_best[m] for m in metric_names], w,
            label=f'threshold={BEST_THRESHOLD:.2f}', color='#378ADD', alpha=0.9)
ax.bar_label(b1, labels=[f'{m_05[m]:.3f}'   for m in metric_names], fontsize=9, padding=3)
ax.bar_label(b2, labels=[f'{m_best[m]:.3f}' for m in metric_names], fontsize=9, padding=3)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.2)
ax.set_ylabel('score — holdout')
ax.set_title('threshold padrao vs otimizado — holdout')
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 curva precision-recall\n\na curva ROC e otimista com classes desbalanceadas porque conta verdadeiros negativos no denominador.  \na curva precision-recall foca exclusivamente nos positivos e e mais informativa nesse cenario.

In [ ]:
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             PrecisionRecallDisplay)

ap = average_precision_score(y_test, y_prob_test)
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, y_prob_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# curva precision-recall
axes[0].plot(rec_curve, prec_curve, color='#378ADD', linewidth=2,
             label=f'XGBoost (AP={ap:.3f})')
axes[0].axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1,
                label=f'chance ({y_test.mean():.2f})')
# marcar threshold otimizado
idx_best = np.argmin(np.abs(thr_curve - BEST_THRESHOLD))
axes[0].scatter(rec_curve[idx_best], prec_curve[idx_best],
                color='#E24B4A', zorder=5, s=80,
                label=f'threshold={BEST_THRESHOLD:.2f}')
axes[0].set_xlabel('recall')
axes[0].set_ylabel('precision')
axes[0].set_title(f'curva precision-recall (AP={ap:.3f})')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# curva ROC
from sklearn.metrics import roc_curve
fpr, tpr, roc_thr = roc_curve(y_test, y_prob_test)
idx_roc = np.argmin(np.abs(roc_thr - BEST_THRESHOLD))
axes[1].plot(fpr, tpr, color='#378ADD', linewidth=2,
             label=f'XGBoost (AUC={auc_test:.3f})')
axes[1].plot([0,1],[0,1], color='gray', linestyle='--', linewidth=1, label='chance')
axes[1].scatter(fpr[idx_roc], tpr[idx_roc], color='#E24B4A', zorder=5, s=80,
                label=f'threshold={BEST_THRESHOLD:.2f}')
axes[1].set_xlabel('FPR (1 - especificidade)')
axes[1].set_ylabel('TPR (recall)')
axes[1].set_title(f'curva ROC (AUC={auc_test:.3f})')
axes[1].legend(fontsize=9)

plt.suptitle('curvas de avaliacao — holdout (n=42)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'average precision (PR-AUC): {ap:.4f}')
print(f'ROC-AUC                   : {auc_test:.4f}')
print(f'nota: PR-AUC < ROC-AUC confirma o desbalanceamento — ROC superestima performance')

### 6.4 resumo consolidado: CV vs holdout

In [ ]:
summary = pd.DataFrame({
    'baseline (CV)': {
        'recall'   : np.array(dummy_cv['test_recall']).mean(),
        'roc_auc'  : np.array(dummy_cv['test_roc_auc']).mean(),
        'f1'       : np.array(dummy_cv['test_f1']).mean(),
        'pr_auc'   : float('nan'),
    },
    'XGBoost CV\n(media 5-fold)': {
        'recall'   : np.mean(cv_scores['recall']),
        'roc_auc'  : np.mean(cv_scores['roc_auc']),
        'f1'       : np.mean(cv_scores['f1']),
        'pr_auc'   : float('nan'),
    },
    f'XGBoost holdout\n(thr=0.50)': {
        'recall'   : m_05['recall'],
        'roc_auc'  : m_05['roc_auc'],
        'f1'       : m_05['f1'],
        'pr_auc'   : float('nan'),
    },
    f'XGBoost holdout\n(thr={BEST_THRESHOLD:.2f})': {
        'recall'   : m_best['recall'],
        'roc_auc'  : m_best['roc_auc'],
        'f1'       : m_best['f1'],
        'pr_auc'   : ap,
    },
}).T

print('=== resumo de metricas ===\n')
print(summary.round(4).to_string())

# heatmap do resumo
fig, ax = plt.subplots(figsize=(8, 4))
mask = summary.isna()
sns.heatmap(summary.astype(float), annot=True, fmt='.3f', cmap='YlGn',
            ax=ax, linewidths=0.5, vmin=0, vmax=1,
            mask=mask, cbar=True, annot_kws={'size': 11})
# celulas com NaN em cinza
sns.heatmap(summary.isna(), annot=False, cmap=['#F0F0F0','#F0F0F0'],
            ax=ax, linewidths=0.5, cbar=False, mask=~mask)
ax.set_title('resumo consolidado de metricas', fontsize=11)
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 7. comparacao: com vs sem `antibiotics_first_2_years`\n\n`antibiotics_first_2_years` e uma covariate do paper original — uso de antibioticos nos primeiros 2 anos de vida.  \nela nao esta disponivel no momento do nascimento (e coletada no follow-up), entao seu papel no modelo precisa ser entendido:\n\n- se adicionar ela aumenta muito o desempenho, o modelo depende de informacao futura para funcionar — limitacao clinica grave\n- se o desempenho for similar sem ela, o modelo tem valor preditivo real desde o nascimento\n\nessa comparacao reproduz a analise do paper original (tabela de sensibilidade) e documenta o impacto da covariate."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "comp0002",
   "metadata": {},
   "source": [
    "### 7.1 treinar XGBoost sem a covariate

In [ ]:
COVARIATE = 'antibiotics_first_2_years'

# features sem a covariate
FEATURES_NO_COV = [f for f in ALL_FEATURES if f != COVARIATE]
cov_idx  = ALL_FEATURES.index(COVARIATE)

X_train_no = np.delete(X_train, cov_idx, axis=1)
X_test_no  = np.delete(X_test,  cov_idx, axis=1)

print(f'features com covariate : {len(ALL_FEATURES)}  {ALL_FEATURES}')
print(f'features sem covariate : {len(FEATURES_NO_COV)}  {FEATURES_NO_COV}')

In [ ]:
def run_cv_xgb(X_tr, y_tr, label):
    scores = {'recall': [], 'roc_auc': [], 'f1': []}

    for tr_idx, va_idx in cv5.split(X_tr, y_tr):
        Xf, Xv = X_tr[tr_idx], X_tr[va_idx]
        yf, yv = y_tr[tr_idx], y_tr[va_idx]

        spw_fold = (yf == 0).sum() / (yf == 1).sum()
        m = XGBClassifier(**best_params_final,
                          scale_pos_weight=spw_fold,
                          eval_metric='logloss',
                          random_state=SEED, n_jobs=-1)
        m.fit(Xf, yf)
        yp = m.predict(Xv)
        yb = m.predict_proba(Xv)[:, 1]

        scores['recall'].append(recall_score(yv, yp, zero_division=0))
        scores['roc_auc'].append(roc_auc_score(yv, yb))
        scores['f1'].append(f1_score(yv, yp, zero_division=0))

    print(f'\n{label}')
    for k, v in scores.items():
        print(f'  {k:<10}: {np.mean(v):.4f}  ± {np.std(v):.4f}')
    return scores

scores_com = run_cv_xgb(X_train,    y_train, 'COM antibiotics_first_2_years')
scores_sem = run_cv_xgb(X_train_no, y_train, 'SEM antibiotics_first_2_years')

### 7.2 comparacao no holdout: com vs sem covariate

In [ ]:
# treinar modelo sem covariate no conjunto de treino completo
xgb_no_cov = XGBClassifier(**best_params_final,
                            scale_pos_weight=spw,
                            eval_metric='logloss',
                            random_state=SEED, n_jobs=-1)
xgb_no_cov.fit(X_train_no, y_train)

y_prob_no  = xgb_no_cov.predict_proba(X_test_no)[:, 1]
y_pred_no  = (y_prob_no >= BEST_THRESHOLD).astype(int)
y_pred_com = (y_prob_test >= BEST_THRESHOLD).astype(int)

holdout_com = metrics_dict(y_test, y_pred_com, y_prob_test)
holdout_sem = metrics_dict(y_test, y_pred_no,  y_prob_no)
holdout_com['pr_auc'] = average_precision_score(y_test, y_prob_test)
holdout_sem['pr_auc'] = average_precision_score(y_test, y_prob_no)

comp2 = pd.DataFrame({
    'COM covariate (holdout)': holdout_com,
    'SEM covariate (holdout)': holdout_sem,
    'delta (com - sem)'      : {k: holdout_com[k] - holdout_sem[k]
                                for k in holdout_com},
}).T

print(f'threshold aplicado: {BEST_THRESHOLD:.2f}\n')
print(comp2.round(4).to_string())

In [ ]:
metrics_plot = ['recall', 'roc_auc', 'f1', 'pr_auc']
x = np.arange(len(metrics_plot))
w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- CV: com vs sem ---
cv_means_com = [np.mean(scores_com[m]) if m in scores_com else float('nan')
                for m in ['recall', 'roc_auc', 'f1', 'pr_auc']]
cv_means_sem = [np.mean(scores_sem[m]) if m in scores_sem else float('nan')
                for m in ['recall', 'roc_auc', 'f1', 'pr_auc']]
cv_stds_com  = [np.std(scores_com[m])  if m in scores_com else 0
                for m in ['recall', 'roc_auc', 'f1', 'pr_auc']]
cv_stds_sem  = [np.std(scores_sem[m])  if m in scores_sem else 0
                for m in ['recall', 'roc_auc', 'f1', 'pr_auc']]

b1 = axes[0].bar(x - w/2, cv_means_com, w, label='com covariate',
                 color='#378ADD', alpha=0.9,
                 yerr=[s if not np.isnan(s) else 0 for s in cv_stds_com], capsize=4)
b2 = axes[0].bar(x + w/2, cv_means_sem, w, label='sem covariate',
                 color='#E24B4A', alpha=0.9,
                 yerr=[s if not np.isnan(s) else 0 for s in cv_stds_sem], capsize=4)
axes[0].bar_label(b1, labels=[f'{v:.3f}' if not np.isnan(v) else '' for v in cv_means_com],
                  fontsize=8, padding=3)
axes[0].bar_label(b2, labels=[f'{v:.3f}' if not np.isnan(v) else '' for v in cv_means_sem],
                  fontsize=8, padding=3)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_plot)
axes[0].set_ylim(0, 1.25)
axes[0].set_title('CV 5-fold (conjunto de treino)')
axes[0].set_ylabel('score medio')
axes[0].legend()

# --- holdout: com vs sem ---
h_com = [holdout_com[m] for m in metrics_plot]
h_sem = [holdout_sem[m] for m in metrics_plot]

b3 = axes[1].bar(x - w/2, h_com, w, label='com covariate', color='#378ADD', alpha=0.9)
b4 = axes[1].bar(x + w/2, h_sem, w, label='sem covariate', color='#E24B4A', alpha=0.9)
axes[1].bar_label(b3, labels=[f'{v:.3f}' for v in h_com], fontsize=8, padding=3)
axes[1].bar_label(b4, labels=[f'{v:.3f}' for v in h_sem], fontsize=8, padding=3)
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_plot)
axes[1].set_ylim(0, 1.25)
axes[1].set_title(f'holdout (thr={BEST_THRESHOLD:.2f})')
axes[1].set_ylabel('score')
axes[1].legend()

plt.suptitle('XGBoost: com vs sem antibiotics_first_2_years', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 7.3 curvas ROC sobrepostas

In [ ]:
fpr_com, tpr_com, _ = roc_curve(y_test, y_prob_test)
fpr_sem, tpr_sem, _ = roc_curve(y_test, y_prob_no)

prec_com, rec_com, _ = precision_recall_curve(y_test, y_prob_test)
prec_sem, rec_sem, _ = precision_recall_curve(y_test, y_prob_no)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr_com, tpr_com, color='#378ADD', linewidth=2,
             label=f'com covariate (AUC={holdout_com["roc_auc"]:.3f})')
axes[0].plot(fpr_sem, tpr_sem, color='#E24B4A', linewidth=2, linestyle='--',
             label=f'sem covariate (AUC={holdout_sem["roc_auc"]:.3f})')
axes[0].plot([0,1],[0,1], color='gray', linestyle=':', linewidth=1, label='chance')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR (recall)')
axes[0].set_title('curva ROC')
axes[0].legend(fontsize=9)

# precision-recall
axes[1].plot(rec_com, prec_com, color='#378ADD', linewidth=2,
             label=f'com covariate (AP={holdout_com["pr_auc"]:.3f})')
axes[1].plot(rec_sem, prec_sem, color='#E24B4A', linewidth=2, linestyle='--',
             label=f'sem covariate (AP={holdout_sem["pr_auc"]:.3f})')
axes[1].axhline(y_test.mean(), color='gray', linestyle=':', linewidth=1,
                label=f'chance ({y_test.mean():.2f})')
axes[1].set_xlabel('recall')
axes[1].set_ylabel('precision')
axes[1].set_title('curva precision-recall')
axes[1].legend(fontsize=9)

plt.suptitle('com vs sem antibiotics_first_2_years — holdout', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 7.4 interpretacao e conclusao da sprint 3\n\n**o que o delta entre com/sem covariate revela:**\n\n- **delta pequeno (< 0.05 em recall e F1):** a covariate tem impacto marginal — o modelo funciona essencialmente com as features do nascimento. Isso e o cenario ideal clinicamente, pois permite predizer o risco antes que o bebe tome antibioticos.\n\n- **delta grande (≥ 0.05):** a covariate explica parte relevante da variancia do target. Nesse caso o modelo com covariate e util para analise retrospectiva e pesquisa, mas nao para triagem neonatal. O modelo sem covariate e o candidato para uso clinico prospectivo.\n\n**nota sobre causalidade:** mesmo que `antibiotics_first_2_years` melhore a performance, ela pode ser um mediador (o cluster ruim leva a mais infeccoes e portanto mais antibioticos) e nao um preditor independente. Incluir mediadores infla artificialmente o AUC sem adicionar valor causal — argumento para reportar ambos os modelos, como o paper original fez.\n\n**proxima sprint (4):** aplicar SHAP sobre o modelo sem covariate para identificar quais features clinicas do nascimento tem maior peso preditivo — essa e a analise de maior valor translacional do projeto."

## 7. salvar modelo final

In [ ]:
import pickle, os, json

out = '../data/processed'

with open(f'{out}/xgb_model1.pkl', 'wb') as f:
    pickle.dump(xgb_final, f)

results = {
    'modelo': 'XGBoost',
    'n_treino': int(len(X_train)),
    'n_holdout': int(len(X_test)),
    'holdout': {
        'recall' : round(rec_test, 4),
        'roc_auc': round(auc_test, 4),
        'f1'     : round(f1_test, 4),
    },
    'cv_media': {
        'recall' : round(np.mean(cv_scores['recall']), 4),
        'roc_auc': round(np.mean(cv_scores['roc_auc']), 4),
        'f1'     : round(np.mean(cv_scores['f1']), 4),
    },
    'best_params': best_params_final,
    'scale_pos_weight': round(spw, 2),
    'features': ALL_FEATURES,
}

with open(f'{out}/model1_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('arquivos salvos:')
for fname in ['xgb_model1.pkl', 'model1_results.json']:
    size = os.path.getsize(f'{out}/{fname}') / 1024
    print(f'  {fname:<28} ({size:.1f} KB)')
print()
print(json.dumps(results, indent=2))